# Урок 18: Забезпечення безпеки агентів ШІ за допомогою криптографічних квитанцій

## Практичний блокнот

Цей блокнот проходить через чотири вправи:

1. **Підпишіть вашу першу квитанцію** для виклику інструмента агента та перевірте її.
2. **Пошкодьте квитанцію** і подивіться, як перевірка не вдасться.
3. **Побудуйте ланцюжок з трьох квитанцій** і підтвердіть цілісність ланцюга.
4. **Оберніть виклик інструмента Microsoft Agent Framework** так, щоб кожна дія породжувала квитанцію.

Всі криптографічні примітиви імпортуються з добре підтримуваних бібліотек (`pynacl` для Ed25519, `jcs` для канонічного JSON за RFC 8785, `hashlib` із стандартної бібліотеки Python для SHA-256). Логіка квитанції сама по собі — це простий Python, який ви можете читати та змінювати.

Запускайте клітинки по черзі. Кожен розділ короткий і самодостатній.


## Налаштування

Встановіть дві залежності. Обидві мають дозволяючі ліцензії (Apache-2.0 / MIT).


In [1]:
%pip install -q pynacl jcs

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import hashlib
import base64
from datetime import datetime, timezone

from nacl import signing
from nacl.exceptions import CryptoError
from jcs import canonicalize

## Допоміжні утиліти

Ці два допоміжні засоби обробляють кодування base64url (без доповнення) та хешування SHA-256 довільних об'єктів. Вони дозволяють залишити іншу частину записника зосередженою на логіці квитанції.


In [3]:
def b64url_nopad(data: bytes) -> str:
    """Base64url-encode bytes without padding (RFC 4648 Section 5)."""
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    """
    Decode unpadded base64url strictly, raising ValueError for anything else.

    Python's decoder skips characters outside the alphabet and also accepts
    padding, the standard +/ alphabet and non-zero spare bits, so the same key
    or signature could be written several ways. Re-encoding and comparing
    accepts only the one form that b64url_nopad produces.
    """
    padding = "=" * ((4 - len(s) % 4) % 4)
    data = base64.urlsafe_b64decode(s + padding)
    if b64url_nopad(data) != s:
        raise ValueError("not canonical unpadded base64url")
    return data

def sha256_canonical(obj) -> str:
    """
    SHA-256 hash of a Python object, computed over its JCS-canonical JSON form.
    Returns a 'sha256:' prefixed hex digest so callers can identify the algorithm.
    """
    canonical = canonicalize(obj)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

## Розділ 1: Підпишіть свій перший квитанційний документ

Уявіть, що наш агента для **Contoso Travel** щойно знайшов рейси з Сіднея до Лос-Анджелеса для клієнта. Ми хочемо зафіксувати заяву шлюзу про цей виклик інструменту, щоб майбутній аудитор міг перевірити її за окремо закріпленим ключем видавця.

### Крок 1.1: Згенеруйте ключ для підпису

У виробництві ключ для підпису агента зазвичай зберігається у модулі апаратної безпеки (HSM), Azure Key Vault або подібному захищеному сховищі. Для цього уроку ми згенеруємо новий ключ у пам’яті.


In [4]:
signing_key = signing.SigningKey.generate()
verify_key = signing_key.verify_key

public_key_b64 = b64url_nopad(bytes(verify_key))
# Trust configuration is out of band. Never trust a receipt only because it embeds a key.
TRUSTED_PUBLIC_KEYS = {public_key_b64}
print(f"Public key (Ed25519, 32 bytes): {public_key_b64}")

Public key (Ed25519, 32 bytes): NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM


### Крок 1.2: Створіть корисне навантаження квитанції

Корисне навантаження містить усе, чому ми хочемо, щоб квитанція засвідчила: хто діяв, який інструмент, з якими аргументами, що повернулося, за якою політикою та коли. Ми хешуємо аргументи та результат замість того, щоб включати їх безпосередньо, щоб квитанція не розкривала конфіденційний вміст.


In [5]:
tool_args = {
    "origin": "SYD",
    "destination": "LAX",
    "departure_date": "2026-06-15",
    "passengers": 2,
}

tool_result = [
    {"flight": "QF11", "price": 1850, "stops": 0},
    {"flight": "UA864", "price": 1620, "stops": 1},
    {"flight": "DL11", "price": 1740, "stops": 0},
]

payload = {
    "type": "agent.tool_call.v1",
    "agent_id": "contoso-travel-bot",
    "tool_name": "lookup_flights",
    "tool_args_hash": sha256_canonical(tool_args),
    "result_hash": sha256_canonical(tool_result),
    "policy_id": "contoso-travel-policy-v3",
    "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sequence": 0,
    "previous_receipt_hash": None,
}

print(json.dumps(payload, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null
}


### Крок 1.3: Підпишіть і зберіть квитанцію

Три кроки:

1. Канонізуйте навантаження за допомогою JCS, щоб дві реалізації, які створюють однакову логічну квитанцію, давали бінарно ідентичні байти.
2. Підпишіть канонічні байти безпосередньо за допомогою приватного ключа Ed25519. PureEdDSA внутрішньо хешує повідомлення, тому додатковий попередній хеш змінив би протокол.

Підпис потім додається до оригінального навантаження, щоб отримати остаточну квитанцію.


In [6]:
def sign_receipt(payload: dict, signing_key: signing.SigningKey, verify_key) -> dict:
    """
    Sign a receipt payload. Returns the receipt with attached signature and public key.
    The 'signature' and 'public_key' fields are NOT part of the canonical signed bytes.
    """
    canonical = canonicalize(payload)
    signature_bytes = signing_key.sign(canonical).signature
    return {
        **payload,
        "signature": {
            "alg": "EdDSA",
            "sig": b64url_nopad(signature_bytes),
            "public_key": b64url_nopad(bytes(verify_key)),
        },
    }

receipt = sign_receipt(payload, signing_key, verify_key)
print(json.dumps(receipt, indent=2))

{
  "type": "agent.tool_call.v1",
  "agent_id": "contoso-travel-bot",
  "tool_name": "lookup_flights",
  "tool_args_hash": "sha256:47578acca4df262c8f172b91493b818a26d042e7beb8e7b121e2bc3776152746",
  "result_hash": "sha256:556447bf01c3c33285086d224b3d6fb4cd7b620b5151fbbebe67e7ff81cdde7a",
  "policy_id": "contoso-travel-policy-v3",
  "timestamp": "2026-10-16T04:44:07Z",
  "sequence": 0,
  "previous_receipt_hash": null,
  "signature": {
    "alg": "EdDSA",
    "sig": "NnH0T6lpjVThg1RS29rEu3OG1FptWbQgX18Yt1JAPjNnTyje2Luozt5_0gcdxddQXoS6eH6Vlg5MTcKLrvWnBQ",
    "public_key": "NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM"
  }
}


### Крок 1.4: Перевірка квитанції

Перевірка є зворотним процесом. Ми видаляємо підпис, повторно обчислюємо канонічні байти і перевіряємо підпис за допомогою відкритого ключа видавця.

Перевірка виконується офлайн: немає викликів сервісу і немає доступу до мережі. Для цього потрібне ще одне, окрім квитанції: відкритий ключ видавця, отриманий через канал, якому аудитор вже довіряє, як-от набір JWK, який публікує видавець, або значення, закріплене заздалегідь. Поле `public_key` у квитанції не може виконувати цю роль, тому що той, хто створює квитанцію, також записує це поле. Розділ 2 демонструє підробку, яку це запобігає.


In [7]:
def verify_receipt(receipt: dict, trusted_public_key: str) -> bool:
    """
    Verify a receipt's Ed25519 signature against a key you already trust.

    trusted_public_key is the issuer's key, obtained from somewhere other than
    this receipt: a JWK Set, a pinned config value, a key registry. Reading the
    key out of receipt["signature"]["public_key"] instead would only prove the
    receipt is consistent with itself. Section 2 shows exactly how that fails.

    It fails closed: anything malformed returns False instead of raising, so
    one bad receipt cannot stop a batch of them from being checked.
    """
    if not isinstance(receipt, dict) or not isinstance(trusted_public_key, str):
        return False
    sig_obj = receipt.get("signature")
    if not isinstance(sig_obj, dict) or sig_obj.get("alg") != "EdDSA":
        return False
    sig_b64 = sig_obj.get("sig")
    if not isinstance(sig_b64, str):
        return False

    try:
        # Reconstruct the payload that was actually signed (everything except signature).
        payload = {k: v for k, v in receipt.items() if k != "signature"}
        canonical = canonicalize(payload)
        verify_key = signing.VerifyKey(b64url_decode(trusted_public_key))
        verify_key.verify(canonical, b64url_decode(sig_b64))
        return True
    except CryptoError:
        return False
    except Exception:
        # An undecodable key or signature, or a payload that cannot be
        # canonicalized: fail closed rather than raise.
        return False

# public_key_b64 came from Section 1, alongside the signing key. In a real
# deployment it would come from the issuer's JWK Set, not from this notebook.
is_valid = verify_receipt(receipt, public_key_b64)
print(f"Receipt is valid: {is_valid}")

# Regression control for the signature scope in draft revision 02. A receipt
# signed over SHA-256(JCS(payload)) is a signature over different bytes and
# must not verify as a direct-JCS Ed25519 receipt.
prehashed_signature = signing_key.sign(hashlib.sha256(canonicalize(payload)).digest()).signature
prehashed_receipt = {
    **receipt,
    "signature": {**receipt["signature"], "sig": b64url_nopad(prehashed_signature)},
}
print(f"Pre-hashed receipt valid: {verify_receipt(prehashed_receipt, public_key_b64)}")

# Malformed input is rejected, not raised.
for malformed in ({"signature": "x"}, {"signature": {"alg": "EdDSA"}}, "not a receipt"):
    assert verify_receipt(malformed, public_key_b64) is False
print("Malformed receipts rejected: True")

Receipt is valid: True
Pre-hashed receipt valid: False
Malformed receipts rejected: True


Ви повинні побачити `Receipt is valid: True`, `Pre-hashed receipt valid: False` та `Malformed receipts rejected: True`. Позитивний випадок демонструє роботу шляху прямого підпису JCS; негативний контроль робить правило області підпису виконуваним замість того, щоб залишати його у вигляді прози; останній рядок показує пошкоджений введення, що повертає `False` замість виключення.


## Розділ 2: Зміна квитанції

Мета квитанцій полягає в тому, що їх неможливо підробити без виявлення. Давайте це доведемо.

Ми змінимо рівно один символ у квитанції та спостерігатимемо, як перевірка не пройде.


In [8]:
import copy

tampered = copy.deepcopy(receipt)

# Modify the policy_id field (this is what an attacker might do to claim
# the action was governed by a more permissive policy than was actually used).
original_policy = tampered["policy_id"]
tampered["policy_id"] = "contoso-travel-policy-PERMISSIVE"

print(f"Original policy_id:  {original_policy}")
print(f"Tampered policy_id:  {tampered['policy_id']}")
print()
print(f"Tampered receipt valid? {verify_receipt(tampered, public_key_b64)}")

Original policy_id:  contoso-travel-policy-v3
Tampered policy_id:  contoso-travel-policy-PERMISSIVE

Tampered receipt valid? False


### Що щойно сталося?

Коли ми змінили `policy_id`, канонічні байти змінилися. Підпис (який був накладений на оригінальні канонічні байти) більше не збігається. Перевірка коректно повертає `False`.

Немає способу змінити підписане поле, тобто будь-що поза обʼєктом `signature`, і при цьому отримати підтвердження квитанції за допомогою публічного ключа видавця, якщо тільки зловмисник не має приватного ключа видавця. Сам обʼєкт `signature` не підписується: `sig` — це підпис, а `verify_receipt` ігнорує `public_key`, тому лише редагування цього поля нічого не змінює. Поки приватний ключ знаходиться у сховищі ключів, а перевіряльники використовують незалежно опублікований публічний ключ видавця, неможливо приховати зміну підписаного поля. Ця умова важлива: наступна клітинка показує змінений квиток, який ідеально перевіряється з іншим ключем.

Спробуйте самі: змініть `tool_name` або `agent_id` чи `timestamp` у вище наведеній клітинці та виконайте знову. Кожна зміна підписаного поля спричиняє недійсний квиток.


### Маніпуляція, яку підпис не помічає

Зазначений вище нападник не перепідписував. Обачніший зробив би так: змінив би квитанцію,
підписав її ключем, який сам згенерував, і вставив цей ключ у поле
`public_key`. Отримана квитанція має підпис, який є повністю справжнім
для її вмісту.

Запустіть це і подивіться, яка з двох перевірок це виявить.


In [9]:
# The attacker generates their own keypair. Nothing stops them.
attacker_key = signing.SigningKey.generate()

forged_payload = {k: v for k, v in receipt.items() if k != "signature"}
forged_payload["policy_id"] = "contoso-travel-policy-PERMISSIVE"

# Re-signed, with the attacker's key embedded where the issuer's used to be.
forged = sign_receipt(forged_payload, attacker_key, attacker_key.verify_key)

print(f"Issuer's public key : {public_key_b64}")
print(f"Key inside the forgery: {forged['signature']['public_key']}")
print()
print(f"Checked against the key inside it: "
      f"{verify_receipt(forged, forged['signature']['public_key'])}")
print(f"Checked against the issuer's key : "
      f"{verify_receipt(forged, public_key_b64)}")

# Assert the results, so the cell fails if the forgery ever passes as the issuer's.
assert verify_receipt(receipt, public_key_b64) is True                    # genuine, issuer's key
assert verify_receipt(forged, forged["signature"]["public_key"]) is True  # forgery, its own key
assert verify_receipt(forged, public_key_b64) is False                    # forgery, issuer's key

Issuer's public key : NKiSqaTHCofGAokp-IyN_y6xJH14J5aUTkL64oMK5uM
Key inside the forgery: nIXlWGm0syJQysGkn-nRlzFS59HlPCTqeQr32TJoHM4

Checked against the key inside it: True
Checked against the issuer's key : False


Ви маєте побачити `True`, потім `False`. Три рядки `assert` роблять це
перевіркою: клітинка викидає помилку, якщо підробка будь-коли підтверджується за допомогою
ключа видавця.

Підробка внутрішньо послідовна. Її підпис справді відповідає
вмісту та ключу, який вона містить, тож перевіряючий, який зчитує ключ із
квитанції, вважає її дійсною. У квитанції нічого не виглядає підозрілим, тому що
нічого в ній не суперечить.

Єдине, що її виявляє — це перевірка за ключем, на який атакувач не міг
вплинути. Ось чому `verify_receipt` приймає ключ як аргумент: функцію з підписом
`verify_receipt(receipt)` не можна написати безпечно,
оскільки єдиний ключ, до якого вона могла б дістатися, — це той, що подав атакувач.

Та сама квитанція зберігається як `sample_receipts/04_resigned_receipt.json`, якщо ви
хочете оглянути її поза ноутбуком.


## Розділ 3: Зв’язування квитанцій у ланцюг

Одна квитанція захищає одну дію. Більшість агентів виконують багато дій. Щоб зробити всю послідовність стійкою до підробки, ми пов’язуємо кожну квитанцію з попередньою, включаючи хеш попередньої квитанції у навантаження нової квитанції.

```text
Receipt 0  -->  Receipt 1  -->  Receipt 2
                  |                 |
                  +-- previous_receipt_hash field --+
```

Якщо хтось видалить або переставить квитанцію всередині представленої послідовності, наступне посилання порушується. Виявлення обрізання або перезапису суфікса, контрольованого підписувачем, також вимагає зафіксованої поточної голови, зовнішньої контрольної точки або автентифікованого зобов’язання населення.


In [10]:
def receipt_hash(receipt: dict) -> str:
    """
    Compute the chain hash of a complete receipt (including signature).
    This becomes the previous_receipt_hash of the next receipt in the chain.
    """
    canonical = canonicalize(receipt)
    digest = hashlib.sha256(canonical).hexdigest()
    return f"sha256:{digest}"

def make_receipt(
    tool_name: str,
    tool_args: dict,
    tool_result,
    sequence: int,
    previous_receipt_hash,
    signing_key,
    verify_key,
    agent_id: str = "contoso-travel-bot",
    policy_id: str = "contoso-travel-policy-v3",
) -> dict:
    """Convenience: build, sign, and return a receipt for one tool call."""
    payload = {
        "type": "agent.tool_call.v1",
        "agent_id": agent_id,
        "tool_name": tool_name,
        "tool_args_hash": sha256_canonical(tool_args),
        "result_hash": sha256_canonical(tool_result),
        "policy_id": policy_id,
        "timestamp": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "sequence": sequence,
        "previous_receipt_hash": previous_receipt_hash,
    }
    return sign_receipt(payload, signing_key, verify_key)

In [11]:
# Build a chain of three receipts: search, hold, book.
r0 = make_receipt(
    tool_name="lookup_flights",
    tool_args={"origin": "SYD", "destination": "LAX", "date": "2026-06-15"},
    tool_result=[{"flight": "QF11", "price": 1850}],
    sequence=0,
    previous_receipt_hash=None,
    signing_key=signing_key,
    verify_key=verify_key,
)

r1 = make_receipt(
    tool_name="hold_seat",
    tool_args={"flight": "QF11", "seat": "14A", "hold_minutes": 30},
    tool_result={"hold_id": "H8472", "expires_at": "2026-06-15T15:00:00Z"},
    sequence=1,
    previous_receipt_hash=receipt_hash(r0),
    signing_key=signing_key,
    verify_key=verify_key,
)

r2 = make_receipt(
    tool_name="confirm_booking",
    tool_args={"hold_id": "H8472", "payment_token": "tok_redacted"},
    tool_result={"booking_ref": "CT-09182", "status": "confirmed"},
    sequence=2,
    previous_receipt_hash=receipt_hash(r1),
    signing_key=signing_key,
    verify_key=verify_key,
)

chain = [r0, r1, r2]
for i, r in enumerate(chain):
    print(f"Receipt {i}: tool={r['tool_name']}, prev={r['previous_receipt_hash']}")

Receipt 0: tool=lookup_flights, prev=None
Receipt 1: tool=hold_seat, prev=sha256:af8b9159b25728881ce6a7166429f4451b9fa869a337453d52228f12131192e7
Receipt 2: tool=confirm_booking, prev=sha256:b1634969232be232f53283251cc9307b443a296909d9e74c2b0101b7f4dc679e


In [12]:
def verify_chain(chain: list, trusted_public_key: str) -> list[dict]:
    """
    Verify a sequence of receipts:
      1. Each receipt's signature must verify.
      2. Each receipt must carry previous_receipt_hash: null for the genesis, and
         the previous receipt's hash for every later receipt.
      3. Each sequence number must be an integer equal to the receipt's zero-based
         position in the chain.
    Returns a list of per-receipt result dicts. A malformed entry fails its own
    checks; it does not stop the rest of the chain from being checked.
    """
    results = []
    for i, receipt in enumerate(chain):
        is_receipt = isinstance(receipt, dict)
        sig_ok = verify_receipt(receipt, trusted_public_key)

        # The link field must be present: .get() alone would read a missing field as None.
        has_link = is_receipt and "previous_receipt_hash" in receipt
        link = receipt.get("previous_receipt_hash") if is_receipt else None
        if i == 0:
            chain_ok = has_link and link is None
        else:
            try:
                chain_ok = has_link and link == receipt_hash(chain[i - 1])
            except Exception:
                chain_ok = False  # the previous entry could not be hashed

        # type() rather than isinstance(): bool is a subclass of int, and True == 1.
        sequence = receipt.get("sequence") if is_receipt else None
        seq_ok = type(sequence) is int and sequence == i
        shape_ok = is_receipt and isinstance(receipt.get("tool_name"), str)

        results.append({
            "index": i,
            "tool": str(receipt.get("tool_name", "?")) if is_receipt else "(not a receipt)",
            "signature_valid": sig_ok,
            "chain_link_valid": chain_ok,
            "sequence_valid": seq_ok,
            "overall_valid": sig_ok and chain_ok and seq_ok and shape_ok,
        })
    return results

for r in verify_chain(chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): VALID
Receipt 2 (   confirm_booking): VALID


Тепер порвіть ланцюжок, змінивши середній квитанцію, і знову перевірте. Змінена квитанція не проходить перевірку підпису, І наступна квитанція не проходить перевірку ланцюжка (оскільки її `previous_receipt_hash` більше не збігається з хешем зміненої середньої квитанції).


In [13]:
# Tamper with the middle receipt: change the hold duration to something
# more permissive than was actually authorized.
tampered_chain = [copy.deepcopy(r) for r in chain]
tampered_chain[1]["tool_args_hash"] = sha256_canonical(
    {"flight": "QF11", "seat": "14A", "hold_minutes": 9999}
)

for r in verify_chain(tampered_chain, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    why = ""
    if not r["overall_valid"]:
        reasons = []
        if not r["signature_valid"]:
            reasons.append("signature")
        if not r["chain_link_valid"]:
            reasons.append("chain link")
        if not r["sequence_valid"]:
            reasons.append("sequence")
        why = " (failed: " + ", ".join(reasons) + ")"
    print(f"Receipt {r['index']} ({r['tool']:>18}): {status}{why}")

Receipt 0 (    lookup_flights): VALID
Receipt 1 (         hold_seat): INVALID (failed: signature)
Receipt 2 (   confirm_booking): INVALID (failed: chain link)


Квитанція 0 все ще проходить перевірку (її не змінювали, і вона не залежить від жодної попередньої). Квитанція 1 не проходить перевірку підпису, оскільки ми змінили `tool_args_hash`. Квитанція 2 не проходить перевірку ланцюгового зв’язку, оскільки її `previous_receipt_hash` була обчислена на основі оригінальної (тепер зміненої) квитанції 1.

Навіть якщо зловмисник повторно підпише змінену квитанцію 1, новий підпис не буде дійсним проти ключа видавця, якщо він не має приватного ключа видавця (повторний підпис власним ключем не вдасться, як показано в Розділі 2), а невідповідність ланцюгового зв’язку в квитанції 2 все одно виявить фальсифікацію. Щоб приховати зміну, зловмиснику довелося б повторно підписати всі квитанції з моменту внесення змін, що потребує володіння приватним ключем видавця.


## Розділ 4: Обгорніть виклик інструменту агента з підписом квитанції

У реальному середовищі розгортання ви не хочете, щоб кожен автор агента запам’ятовував виклик `make_receipt`. Ви хочете, щоб підпис квитанції був автоматичним для кожного виклику інструменту.

Ось найпростіший шаблон: клас-обгортка, який приймає будь-яку викликану функцію інструменту і повертає версію з генерацією квитанції. Це адаптується до будь-якого агентського фреймворку, включно з Microsoft Agent Framework (`agent_framework.foundry`).

Якщо у вас немає встановленого проекту Microsoft Foundry, наведену нижче локальну емуляцію все одно демонструє цей шаблон.


In [14]:
class ReceiptedTool:
    """
    Wraps a tool function so every invocation produces a signed receipt.
    Receipts are appended to a chain held by this object.

    Accepts both positional and keyword arguments. The receipt's
    tool_args field records args (as a list) and kwargs (as a dict)
    so the canonical hash binds to whichever the caller supplied.
    """

    def __init__(self, name: str, fn, signing_key, verify_key, agent_id: str, policy_id: str):
        self.name = name
        self.fn = fn
        self.signing_key = signing_key
        self.verify_key = verify_key
        self.agent_id = agent_id
        self.policy_id = policy_id
        self.receipts: list = []

    def __call__(self, *args, **kwargs):
        result = self.fn(*args, **kwargs)
        previous_hash = receipt_hash(self.receipts[-1]) if self.receipts else None
        receipt = make_receipt(
            tool_name=self.name,
            tool_args={"args": list(args), "kwargs": kwargs},
            tool_result=result,
            sequence=len(self.receipts),
            previous_receipt_hash=previous_hash,
            signing_key=self.signing_key,
            verify_key=self.verify_key,
            agent_id=self.agent_id,
            policy_id=self.policy_id,
        )
        self.receipts.append(receipt)
        return result

In [15]:
# Example tool: a mock flight lookup. In a real Microsoft Agent Framework deployment,
# this would be a function passed to FoundryChatClient as a tool.
def mock_lookup_flights(origin: str, destination: str, departure_date: str) -> list:
    return [
        {"flight": "QF11", "price": 1850, "stops": 0},
        {"flight": "UA864", "price": 1620, "stops": 1},
    ]

# Wrap it with receipt signing.
receipted_lookup = ReceiptedTool(
    name="lookup_flights",
    fn=mock_lookup_flights,
    signing_key=signing_key,
    verify_key=verify_key,
    agent_id="contoso-travel-bot",
    policy_id="contoso-travel-policy-v3",
)

# Use the wrapped tool exactly like the original.
results_a = receipted_lookup(origin="SYD", destination="LAX", departure_date="2026-06-15")
results_b = receipted_lookup(origin="SYD", destination="NRT", departure_date="2026-07-02")
results_c = receipted_lookup(origin="MEL", destination="SIN", departure_date="2026-08-10")

print(f"Tool was called {len(receipted_lookup.receipts)} times.")
print(f"Each call produced a signed receipt linked to the previous one.")
print()

for r in verify_chain(receipted_lookup.receipts, public_key_b64):
    status = "VALID" if r["overall_valid"] else "INVALID"
    print(f"Receipt {r['index']} ({r['tool']}): {status}")


Tool was called 3 times.
Each call produced a signed receipt linked to the previous one.

Receipt 0 (lookup_flights): VALID
Receipt 1 (lookup_flights): VALID
Receipt 2 (lookup_flights): VALID


### Інтеграція з Microsoft Agent Framework

Обгортка `ReceiptedTool` вище є незалежною від фреймворку. Щоб використовувати її всередині агента, створеного з Microsoft Agent Framework, зареєструйте обгорнуту функцію як інструмент. Ескіз (ви заміните макет на реєстрацію інструменту Microsoft Foundry):

```python
# Pseudocode showing the integration shape.
# import os
# from agent_framework.foundry import FoundryChatClient
# from azure.identity import AzureCliCredential
#
# provider = FoundryChatClient(
#     project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
#     model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
#     credential=AzureCliCredential(),
# )
# agent = provider.as_agent(
#     instructions="You are a Contoso Travel agent ...",
#     tools=[receipted_lookup],   # the wrapped tool, not the raw function
# )
# response = agent.run("Find flights from Sydney to Los Angeles in June.")
#
# # After the run, every tool call the agent made has a signed receipt:
# audit_chain = receipted_lookup.receipts
```

Фреймворку агента не потрібно нічого знати про квитанції. Підписання квитанції обгорнуте навколо інструменту, а не вбудоване у фреймворк. Ось як ви додаєте слід походження до існуючого коду агента без переписування агента.


## Recap and stretch challenge

You have:

- Generated an Ed25519 key pair.
- Built and signed a receipt for an agent tool call.
- Verified the receipt offline using only the issuer's public key, obtained independently of the receipt.
- Tampered with a receipt and observed verification fail.
- Built a hash-chained sequence of three receipts.
- Tampered with the middle of the chain and observed both signature failure and chain-link failure.
- Wrapped a tool function with automatic receipt signing.

**Stretch challenge.** Extend the receipt schema with a `request_id` field (a UUID for distributed tracing). Update `make_receipt` to include it, and confirm receipts still verify end to end. Then modify the field after signing and confirm verification fails. This forces you to internalize how every byte of the canonical encoding contributes to the signature.

**Important boundary.** Under a pinned key and chain context, receipts prove signature attribution, content integrity, and a link to the presented predecessor. They do NOT prove execution, source truth, trusted time, completeness, policy evaluation, or correctness. Receipts are a foundation. Governance is the system you build on top.

Read the lesson README again with that boundary in mind. The most common mistake teams make with receipts is assuming "we have receipts" means "we are governed." It does not. Receipts make agent behavior auditable. They do not make it correct.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Відмова від відповідальності**:
Цей документ було перекладено за допомогою сервісу штучного інтелекту для перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, будь ласка, майте на увазі, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ рідною мовою слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується професійний людський переклад. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
